In [2]:
from google.colab import drive, auth
import tensorflow as tf
import os, subprocess
import random

drive.mount('/content/drive')
auth.authenticate_user()

# Confirm GPU
gpus = tf.config.list_physical_devices('GPU')
print(f"GPU: {gpus}")
assert len(gpus) > 0, "No GPU found. Check runtime type."

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

strategy     = tf.distribute.MirroredStrategy()
GLOBAL_BATCH = 32
IMG_SIZE     = (224, 224, 3)
N_CLASSES    = 3
AUTOTUNE     = tf.data.AUTOTUNE

GCS_TFRECORDS = 'gs://tala-sentinel2-data/tfrecords/sentinel2'
GCS_MODELS    = 'gs://tala-sentinel2-data/models/proxy_cnn'
GCS_LOGS      = 'gs://tala-sentinel2-data/logs/proxy_cnn'

for d in [GCS_MODELS, GCS_LOGS]:
    subprocess.run(
        ['gsutil', '-q', 'cp', '/dev/null', f'{d}/.keep'],
        capture_output=True
    )

print(f"Strategy     : {strategy.__class__.__name__}")
print(f"Global batch : {GLOBAL_BATCH}")
print(f"TF version   : {tf.__version__}")
print("✓ Setup complete")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Strategy     : MirroredStrategy
Global batch : 32
TF version   : 2.19.0
✓ Setup complete


In [7]:
import random

feature_spec = {
    'image'     : tf.io.FixedLenFeature([], tf.string),
    'label'     : tf.io.FixedLenFeature([], tf.int64),
    'cluster_id': tf.io.FixedLenFeature([], tf.int64),
    'quarter'   : tf.io.FixedLenFeature([], tf.int64),
    'height'    : tf.io.FixedLenFeature([], tf.int64),
    'width'     : tf.io.FixedLenFeature([], tf.int64),
    'channels'  : tf.io.FixedLenFeature([], tf.int64),
}

def parse_tfrecord(example_proto):
    parsed = tf.io.parse_single_example(example_proto, feature_spec)
    h   = tf.cast(parsed['height'],   tf.int32)
    w   = tf.cast(parsed['width'],    tf.int32)
    c   = tf.cast(parsed['channels'], tf.int32)

    img = tf.io.decode_raw(parsed['image'], tf.float32)
    img = tf.reshape(img, [h, w, c])

    # 1. Leave as [0.0, 1.0] for now so augmentations work!
    return img, tf.cast(parsed['label'], tf.int32)

def filter_nans(img, label):
    # 2. Return True only if there are NO NaNs in the image
    return tf.math.logical_not(tf.reduce_any(tf.math.is_nan(img)))

def augment(img, label):
    # 3. Augmentations work safely in the [0.0, 1.0] space
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, max_delta=0.1)
    img = tf.image.random_contrast(img, lower=0.9, upper=1.1)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, label

def preprocess_vgg(img, label):
    # 4. Multiply by 255 and apply VGG mean subtraction LAST
    img = img * 255.0
    img = tf.keras.applications.vgg16.preprocess_input(img)
    return img, label

def build_dataset(split='train', repeat=False):
    all_files = sorted(
        tf.io.gfile.glob(f'{GCS_TFRECORDS}/*.tfrecord'))

    random.Random(42).shuffle(all_files)

    n_train = int(len(all_files) * 0.8)
    files   = (all_files[:n_train]
               if split == 'train'
               else all_files[n_train:])

    print(f"  {split}: {len(files)} shards")

    ds = tf.data.TFRecordDataset(
        files, num_parallel_reads=AUTOTUNE)
    ds = ds.map(parse_tfrecord,
                num_parallel_calls=AUTOTUNE)
    ds = ds.filter(filter_nans)

    if split == 'train':
        ds = ds.map(augment,
                    num_parallel_calls=AUTOTUNE)
        ds = ds.shuffle(1000,
                        reshuffle_each_iteration=True)

    ds = ds.map(preprocess_vgg,
                num_parallel_calls=AUTOTUNE)
    ds = ds.batch(GLOBAL_BATCH, drop_remainder=False)

    # repeat() makes the dataset loop indefinitely
    # so Keras never runs out mid-epoch
    if repeat:
        ds = ds.repeat()

    ds = ds.prefetch(AUTOTUNE)
    return ds

# ── COUNT ACTUAL BATCHES FIRST ─────────────────────────
# Do this ONCE to get the real batch count after filtering
# Takes about 1 minute
print("Counting batches after NaN filter...")
print("(Do this once, then use the numbers below)")

train_ds_count = build_dataset('train', repeat=False)
val_ds_count   = build_dataset('val',   repeat=False)

train_steps = sum(1 for _ in train_ds_count)
val_steps   = sum(1 for _ in val_ds_count)

print(f"Train batches : {train_steps}")
print(f"Val batches   : {val_steps}")

# Now rebuild with repeat=True for actual training
print("\nRebuilding datasets with repeat...")
train_ds = build_dataset('train', repeat=True)
val_ds   = build_dataset('val',   repeat=True)

# Verify
for imgs, labels in train_ds.take(1):
    print(f"Batch shape : {imgs.shape}")
    print(f"Pixel range : [{imgs.numpy().min():.3f},"
          f" {imgs.numpy().max():.3f}]")
    print(f"Labels      : {labels.numpy()[:8]}")
print("✓ Dataset ready")
print(f"\nUse these in model.fit():")
print(f"  steps_per_epoch={train_steps}")
print(f"  validation_steps={val_steps}")

Counting batches after NaN filter...
(Do this once, then use the numbers below)
  train: 26 shards
  val: 7 shards
Train batches : 105
Val batches   : 28

Rebuilding datasets with repeat...
  train: 26 shards
  val: 7 shards
Batch shape : (32, 224, 224, 3)
Pixel range : [-123.680, 151.061]
Labels      : [1 1 0 2 0 2 1 1]
✓ Dataset ready

Use these in model.fit():
  steps_per_epoch=105
  validation_steps=28


In [8]:
with strategy.scope():
    base = tf.keras.applications.VGG16(
        weights     = 'imagenet',
        include_top = False,
        input_shape = IMG_SIZE
    )
    base.trainable = False

    inputs  = tf.keras.Input(shape=IMG_SIZE)
    x       = base(inputs, training=False)
    x       = tf.keras.layers.GlobalAveragePooling2D()(x)
    x       = tf.keras.layers.Dense(
                  4096, activation='relu',
                  name='feature_vector')(x)
    x       = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(
                  N_CLASSES, activation='softmax')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer = tf.keras.optimizers.Adam(1e-4),
        loss      = 'sparse_categorical_crossentropy',
        metrics   = ['accuracy']
    )

model.summary()

# ── PHASE A: HEAD ONLY ─────────────────────────────────
print("\n" + "="*50)
print("PHASE A: Training head only")
print("="*50)

callbacks_a = [
    tf.keras.callbacks.EarlyStopping(
        monitor              = 'val_accuracy',
        patience             = 5,
        restore_best_weights = True,
        verbose              = 1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath       = f'{GCS_MODELS}/phase_a_best.keras',
        monitor        = 'val_accuracy',
        save_best_only = True,
        verbose        = 1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor  = 'val_accuracy',
        factor   = 0.5,
        patience = 3,
        min_lr   = 1e-7,
        verbose  = 1
    ),
    tf.keras.callbacks.TensorBoard(
        log_dir     = f'{GCS_LOGS}/phase_a',
        update_freq = 'epoch'
    )
]

history_a = model.fit(
    train_ds,
    validation_data = val_ds,
    epochs          = 15,
    steps_per_epoch  = train_steps,      # ADD THIS
    validation_steps = val_steps,        # ADD THIS
    callbacks       = callbacks_a,
    verbose         = 1
)

best_a = max(history_a.history['val_accuracy'])
print(f"\nPhase A complete. Best val_accuracy: {best_a:.4f}")

# ── PHASE B: FINE-TUNE TOP CONV BLOCKS ────────────────
print("\n" + "="*50)
print("PHASE B: Fine-tuning top VGG16 block")
print("="*50)

with strategy.scope():
    base.trainable = True
    for layer in base.layers[:-4]:
        layer.trainable = False

    trainable = [l.name for l in base.layers
                 if l.trainable]
    print(f"Unfrozen: {trainable}")

    model.compile(
        optimizer = tf.keras.optimizers.Adam(1e-5),
        loss      = 'sparse_categorical_crossentropy',
        metrics   = ['accuracy']
    )

callbacks_b = [
    tf.keras.callbacks.EarlyStopping(
        monitor              = 'val_accuracy',
        patience             = 7,
        restore_best_weights = True,
        verbose              = 1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath       = f'{GCS_MODELS}/phase_b_best.keras',
        monitor        = 'val_accuracy',
        save_best_only = True,
        verbose        = 1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor  = 'val_accuracy',
        factor   = 0.5,
        patience = 4,
        min_lr   = 1e-8,
        verbose  = 1
    ),
    tf.keras.callbacks.TensorBoard(
        log_dir     = f'{GCS_LOGS}/phase_b',
        update_freq = 'epoch'
    )
]

history_b = model.fit(
    train_ds,
    validation_data = val_ds,
    epochs          = 30,
    steps_per_epoch  = train_steps,      # ADD THIS
    validation_steps = val_steps,        # ADD THIS
    callbacks       = callbacks_b,
    verbose         = 1
)

best_b = max(history_b.history['val_accuracy'])
print(f"\nPhase B complete. Best val_accuracy: {best_b:.4f}")

# ── SAVE ───────────────────────────────────────────────
final_path = f'{GCS_MODELS}/proxy_cnn_final.keras'
model.save(final_path)
print(f"Model saved: {final_path}")

# Backup to Drive
import os
os.makedirs('/content/drive/MyDrive/Models', exist_ok=True)
subprocess.run([
    'gsutil', 'cp', final_path,
    '/content/drive/MyDrive/Models/proxy_cnn_final.keras'
])
print("Backup saved to Drive")

# ── SUMMARY ────────────────────────────────────────────
print("\n" + "="*50)
print("TRAINING COMPLETE")
print("="*50)
print(f"Phase A best val_accuracy : {best_a:.4f}")
print(f"Phase B best val_accuracy : {best_b:.4f}")
print()
print("Interpreting results:")
print("  > 0.80  excellent")
print("  > 0.65  good, proceed confidently")
print("  > 0.50  acceptable, features still useful")
print("  < 0.50  re-run Phase B with more layers unfrozen")
print()
print("Next: open feature extraction notebook (GPU runtime)")
print(f"Model: {final_path}")

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_vector (Dense)          │ (None, 4096)           │     2,101,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │        12,291 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,828,227 (64.19 MB)

 Trainable params: 2,113,539 (8.06 MB)

 Non-trainable params: 14,714,688 (56.13 MB)


PHASE A: Training head only
Epoch 1/15
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 583ms/step - accuracy: 0.5924 - loss: 1.0051
Epoch 1: val_accuracy improved from None to 0.67506, saving model to gs://tala-sentinel2-data/models/proxy_cnn/phase_a_best.keras

Epoch 1: finished saving model to gs://tala-sentinel2-data/models/proxy_cnn/phase_a_best.keras
105/105 ━━━━━━━━━━━━━━━━━━━━ 116s 867ms/step - accuracy: 0.6348 - loss: 0.8953 - val_accuracy: 0.6751 - val_loss: 0.7548 - learning_rate: 1.0000e-04
Epoch 2/15
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 558ms/step - accuracy: 0.7170 - loss: 0.6652
Epoch 2: val_accuracy improved from 0.67506 to 0.67849, saving model to gs://tala-sentinel2-data/models/proxy_cnn/phase_a_best.keras

Epoch 2: finished saving model to gs://tala-sentinel2-data/models/proxy_cnn/phase_a_best.keras
105/105 ━━━━━━━━━━━━━━━━━━━━ 83s 793ms/step - accuracy: 0.7127 - loss: 0.6782 - val_accuracy: 0.6785 - val_loss: 0.7480 - learning_rate: 1.0000e-04
Epoch 3/15
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 